In [4]:
import importlib
for pkg in ['openreview','pandas']:
    assert importlib.util.find_spec(pkg) is not None, f'Missing package: {pkg}'
print('Dependencies available in current kernel environment.')


Dependencies available in current kernel environment.


In [5]:
import openreview
import pandas as pd
import os
import time

In [6]:
os.makedirs('data/openreview', exist_ok=True)

client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net'
)

VENUE_ID = 'ICLR.cc/2024/Conference'
LIMIT = 7000  # increase if you want more
OUT_DIR = 'data/openreview'


In [7]:
submissions = client.get_all_notes(
    invitation=f"{VENUE_ID}/-/Submission"
)

print("submissions:", len(submissions))

submissions: 7404


In [8]:
rows = []

target = min(LIMIT, len(submissions))
for i, paper in enumerate(submissions[:target], start=1):
    print(f'Processing {i}/{target}: paper {paper.number}')

    content = paper.content or {}

    title = (content.get('title') or {}).get('value') if isinstance(content.get('title'), dict) else content.get('title')
    abstract = (content.get('abstract') or {}).get('value') if isinstance(content.get('abstract'), dict) else content.get('abstract')
    keywords = (content.get('keywords') or {}).get('value') if isinstance(content.get('keywords'), dict) else content.get('keywords')
    pdf = (content.get('pdf') or {}).get('value') if isinstance(content.get('pdf'), dict) else content.get('pdf')

    replies = client.get_all_notes(forum=paper.id)
    time.sleep(0.1)

    reviews = []
    decisions = []

    for r in replies:
        invitation = r.invitations[0] if r.invitations else ''
        r_content = r.content or {}

        if 'Official_Review' in invitation:
            reviews.append({
                'rating': (r_content.get('rating') or {}).get('value') if isinstance(r_content.get('rating'), dict) else r_content.get('rating'),
                'confidence': (r_content.get('confidence') or {}).get('value') if isinstance(r_content.get('confidence'), dict) else r_content.get('confidence'),
                'summary': (r_content.get('summary') or {}).get('value') if isinstance(r_content.get('summary'), dict) else r_content.get('summary'),
                'strengths': (r_content.get('strengths') or {}).get('value') if isinstance(r_content.get('strengths'), dict) else r_content.get('strengths'),
                'weaknesses': (r_content.get('weaknesses') or {}).get('value') if isinstance(r_content.get('weaknesses'), dict) else r_content.get('weaknesses')
            })

        if 'Decision' in invitation:
            decisions.append({
                'decision': (r_content.get('decision') or {}).get('value') if isinstance(r_content.get('decision'), dict) else r_content.get('decision'),
                'comment': (r_content.get('comment') or {}).get('value') if isinstance(r_content.get('comment'), dict) else r_content.get('comment')
            })

    rows.append({
        'paper_id': paper.id,
        'number': paper.number,
        'title': title,
        'abstract': abstract,
        'keywords': keywords,
        'pdf': pdf,
        'n_reviews': len(reviews),
        'reviews': reviews,
        'decision': decisions[0]['decision'] if decisions else None,
        'decision_comment': decisions[0]['comment'] if decisions else None
    })


Processing 1/7000: paper 9504
Processing 2/7000: paper 9502
Processing 3/7000: paper 9498
Processing 4/7000: paper 9493
Processing 5/7000: paper 9491
Processing 6/7000: paper 9489
Processing 7/7000: paper 9483
Processing 8/7000: paper 9482
Processing 9/7000: paper 9481
Processing 10/7000: paper 9480
Processing 11/7000: paper 9477
Processing 12/7000: paper 9476
Processing 13/7000: paper 9472
Processing 14/7000: paper 9468
Processing 15/7000: paper 9467
Processing 16/7000: paper 9466
Processing 17/7000: paper 9463
Processing 18/7000: paper 9460
Processing 19/7000: paper 9459
Processing 20/7000: paper 9458
Processing 21/7000: paper 9456
Processing 22/7000: paper 9455
Processing 23/7000: paper 9453
Processing 24/7000: paper 9451
Processing 25/7000: paper 9450
Processing 26/7000: paper 9446
Processing 27/7000: paper 9441
Processing 28/7000: paper 9439
Processing 29/7000: paper 9438
Processing 30/7000: paper 9437
Processing 31/7000: paper 9436
Processing 32/7000: paper 9434
Processing 33/700

In [9]:
df = pd.DataFrame(rows)

print(df.shape)
df.head(10)

(7000, 10)


,paper_id,number,title,abstract,keywords,pdf,n_reviews,reviews,decision,decision_comment
0,cXs5md5wAq,9504,Modelling Microbial Communities with Graph Neu...,Understanding the interactions and interplay o...,"[graph neural networks, microbial communities,...",/pdf/a4578db3b369ee02db6e42b64d333e578e1b692e.pdf,4,"[{'rating': '3: reject, not good enough', 'con...",Reject,
1,rhgIgTSSxW,9502,TabR: Tabular Deep Learning Meets Nearest Neig...,Deep learning (DL) models for tabular data pro...,"[tabular, tabular data, architecture, deep lea...",/pdf/178e173a880d7872c0a79d88e005426c20501329.pdf,4,"[{'rating': '8: accept, good paper', 'confiden...",Accept (poster),
2,kKRbAY4CXv,9498,Neural Evolutionary Kernel Method: A Knowledge...,Numerical solution of partial differential equ...,"[Numerical PDE, structure preserving neural ne...",/pdf/c330ae354c1b65e4afaa1f53a1ca188d24fcf27f.pdf,4,[{'rating': '6: marginally above the acceptanc...,Reject,
3,ApjY32f3Xr,9493,PINNacle: A Comprehensive Benchmark of Physics...,While significant progress has been made on Ph...,"[PINN, machine learning, physics-informed mach...",/pdf/49840b2f19f2bbeff0c1539d86c876d01140da89.pdf,4,[{'rating': '6: marginally above the acceptanc...,Reject,
4,eUgS9Ig8JG,9491,SaNN: Simple Yet Powerful Simplicial-aware Neu...,Simplicial neural networks (SNNs) are deep mod...,"[Graph Neural Networks, Higher-order Represent...",/pdf/b5b2e785dec69b9ea0c8b01d6e2eca5896246cce.pdf,4,"[{'rating': '8: accept, good paper', 'confiden...",Accept (spotlight),
5,mnyXZBa5dP,9489,Image Authenticity Detection using Eye Gazing ...,"In the digital age, determining the authentici...","[Image Manipulation Detection, Cascade Network...",None,0,[],None,None
6,fMX07g3prp,9483,FR-NAS: Forward-and-Reverse Graph Predictor fo...,Neural Architecture Search (NAS) has risen to ...,"[Neural Architecture Search, Performance Predi...",/pdf/7f9544f40910418069f642bcc32bcae5906ff521.pdf,4,"[{'rating': '3: reject, not good enough', 'con...",None,None
7,qBL04XXex6,9482,Boosting of Thoughts: Trial-and-Error Problem ...,The reasoning performance of Large Language Mo...,[Large Language Models; Prompt Engineering; Bo...,/pdf/a30673a601700226be14c851d887cc7181f4c78f.pdf,5,[{'rating': '5: marginally below the acceptanc...,Accept (poster),
8,H9DYMIpz9c,9481,Farzi Data: Autoregressive Data Distillation,We study data distillation for auto-regressive...,"[Data Distillation, Meta Learning, Recommender...",/pdf/c2039562a8f9e0be209b8a58cce8414264bb7a48.pdf,5,[{'rating': '6: marginally above the acceptanc...,Reject,
9,rp5vfyp5Np,9480,BATTLE: Towards Behavior-oriented Adversarial ...,Evaluating the performance of deep reinforceme...,"[deep reinforcement learning, preference-based...",/pdf/bd31536c071f627ae09428e0b7506a25d0824163.pdf,4,[{'rating': '5: marginally below the acceptanc...,Reject,


In [10]:
import os
import pandas as pd

df = pd.DataFrame(rows)
os.makedirs(OUT_DIR, exist_ok=True)
out_path = f'{OUT_DIR}/iclr2024_openreview_{len(df)}.csv'
df.to_csv(out_path, index=False, encoding='utf-8')
print(f'Saved: {out_path} | shape={df.shape}')


Saved: data/openreview/iclr2024_openreview_7000.csv | shape=(7000, 10)


In [11]:
print("rows collected:", len(rows))
print("csv rows:", len(df))

rows collected: 7000
csv rows: 7000


In [12]:
import os
import json

os.makedirs(OUT_DIR, exist_ok=True)
out_json = f'{OUT_DIR}/iclr2024_openreview_{len(rows)}.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f'Saved: {out_json} | rows={len(rows)}')


Saved: data/openreview/iclr2024_openreview_7000.json | rows=7000


In [13]:
if len(df) > 0:
    pdf_path = df.loc[0, 'pdf']
    url = pdf_path if str(pdf_path).startswith('http') else 'https://openreview.net' + str(pdf_path)
    print('Sample PDF URL:', url)
else:
    print('No rows collected.')


Sample PDF URL: https://openreview.net/pdf/a4578db3b369ee02db6e42b64d333e578e1b692e.pdf
